# Lesson 9 — File Handling

Persistence: reading, writing, appending — and surviving missing files.

## How to work

- Write every solution in `solution.py` first, then paste the finished
  code into the matching cell and run it.
- **Rule for every exercise:** your printed output must match the
  `Output:` block **exactly**. File contents are output too.
- Every cell builds its own fixture files first — cells are
  self-contained; re-running a cell must reproduce its output exactly.
- Read the `Constraint:` line. Violating it is a fail, even if the
  output looks right.
- Every `open` uses a `with` block and an explicit
  `encoding="utf-8"` unless a constraint says otherwise.
- Functions never print unless the task says so; printing happens in
  the driver code.
- Hints are conceptual keywords, not answers — ask for one at a time.
- H1 ships broken code on purpose: find and fix the bugs; don't rewrite
  working parts.

Difficulty climbs after the Easy set.

## E1 — Round trip

**Type:** [REAL-WORLD]

**Task:** write two small functions and run the driver below:
`save_greeting(path)` writes exactly two lines to the file —
`hello` and `file handling`; `load_text(path)` returns the file's whole
content as one string.

```python
save_greeting("greetings.txt")
print(load_text("greetings.txt"), end="")
```

**Output:**
```
hello
file handling
```

**Constraint:** both functions use `with` + explicit utf-8. Exactly two
`write` calls build the file (each line ends with `\n`). Loading uses
`.read()` once.

**Hints (ask one at a time):**
- The printed output ends without a gap because the driver suppressed print's own newline — who supplied the final one?
- Two lines of content means two newline characters in the file. Where does each come from?
- `.read()` hands back everything including those newlines.

In [37]:
def save_greeting(path):
    with open(path, "w", encoding="utf-8") as f:
        f.write("hello\n")
        f.write("file handling\n")
    
def load_text(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"file {path} not found"
    
save_greeting("greetings.txt")
print(load_text("greetings.txt"), end="")

hello
file handling


## E2 — Append proves memory

**Type:** [REAL-WORLD]

**Task:** write `append_line(path, line)` adding one line to a file,
then run the driver:

```python
with open("events.log", "w", encoding="utf-8") as f:
    pass                      # start from an empty file

append_line("events.log", "boot")
append_line("events.log", "ready")
append_line("events.log", "shutdown")

print(load_text("events.log"), end="")
```

**Output:**
```
boot
ready
shutdown
```

(Reuse your E1 `load_text` — pasting it into this cell is fine.)

**Constraint:** `append_line` opens in append mode. Opening in any
write-truncating mode makes only the last call survive.

**Hints (ask one at a time):**
- Three separate calls, three separate openings — which mode keeps earlier calls alive?
- One mode parks the cursor at the end before every write.
- The driver's empty `with open(..., "w")` is deliberate: what would happen without it if the file existed?

In [76]:
def append_line(path, line):
    with open(path, "a", encoding="utf-8") as f:
        f.write(line + "\n")
            
with open("events.log", "w", encoding="utf-8"):
    pass

append_line("events.log", "boot")
append_line("events.log", "ready")
append_line("events.log", "shutdown")

print(load_text("events.log"), end="")

boot
ready
shutdown


## E3 — One line at a time

**Type:** [REAL-WORLD]

**Task:** write `report_temperatures(path)` that iterates a file of
one-number-per-line and returns `(count, total)`. Driver:

```python
with open("temperatures.txt", "w", encoding="utf-8") as f:
    f.write("21.5\n20.1\n22.8\n")

c, t = report_temperatures("temperatures.txt")
print(f"lines: {c}")
print(f"total: {t:.1f}")
```

**Output:**
```
lines: 3
total: 64.4
```

**Constraint:** iterate the file object directly (`for line in f`). No
`read()`, no `readlines()`. Convert inside the loop — `float()`
tolerates the trailing newline.

**Hints (ask one at a time):**
- Two accumulators, one loop — where were they initialized in Lesson 4's pattern?
- The handle itself is the sequence; each step hands you one line as a string.
- You've built this exact count-and-total loop twice before — only the data source changed.

In [19]:
def report_temperatures(path):
    count_lines = 0
    total = 0
    
    try:
        with open(path, encoding="utf-8") as f:
            for line in f:
                count_lines += 1
                total += float(line)
        return count_lines, total
    except FileNotFoundError:
        return count_lines, total

with open("temperatures.txt", "w", encoding="utf-8") as f:
    f.write("21.5\n20.1\n22.8\n")
    
c, t = report_temperatures("temperatures.txt")
print(f"lines: {c}")
print(f"total: {t:.1f}")
            

lines: 3
total: 64.4


## E4 — The file wasn't there

**Type:** [REAL-WORLD]

**Task:** write `load_greeting(path)` returning the file's content, or
the default string `hello, newcomer` when no file exists. Driver:

```python
print(load_greeting("user-name.txt"))

with open("user-name.txt", "w", encoding="utf-8") as f:
    f.write("hello, Ada")

print(load_greeting("user-name.txt"))
```

**Output:**
```
hello, newcomer
hello, Ada
```

**Constraint:** one function serves both calls. Catch the specific
missing-file exception only. The `with` lives inside the `try`.

**Hints (ask one at a time):**
- First call: the file doesn't exist yet — which exception fires at open time?
- The fallback is just a return value from the except block — nothing fancier.
- Second call must take the success path through the same function — no special-casing outside.

In [75]:
def load_greeting(path):
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return "hello, newcomer"
    
print(load_greeting("user-name.txt"))

with open("user-name.txt", "w", encoding="utf-8") as f:
    f.write("hello, Ada")
    
print(load_greeting("user-name.txt"))

hello, newcomer
hello, Ada


## E5 — Report writer

**Type:** [REAL-WORLD]

**Task:** write `write_report(stock, path)` that writes one
`name: qty` line per dict entry, then the driver prints the file back:

```python
stock = {"apples": 12, "bananas": 0, "cherries": 7}

write_report(stock, "report.txt")
print(load_text("report.txt"), end="")
```

**Output:**
```
apples: 12
bananas: 0
cherries: 7
```

**Constraint:** build all lines first, then ONE `write` of the joined
text plus the final newline. Dict order preserved. The writer never
prints.

**Hints (ask one at a time):**
- A comprehension can shape every line first; joining comes after.
- Joining N lines needs N-1 newlines — what closes the last line?
- Iterating a dict hands you keys — how do you reach the quantity?

In [57]:
def write_report(stock: dict, path):
    with open(path, "w", encoding="utf-8") as f:
        report = [f"{item}: {qty}\n" for item, qty in stock.items()]
        f.write("".join(report))
            
stock = {"apples": 12, "bananas": 0, "cherries": 7}

write_report(stock, "report.txt")
print(load_text("report.txt"), end="")

apples: 12
bananas: 0
cherries: 7


## M1 — Transactions file → summary

**Type:** [REAL-WORLD]

**Task:** same record grammar you validated in Lesson 8 — lines of
`name,amount`, amount parseable by `float` (leading `+`/`-` allowed),
name non-empty; invalid = wrong shape or unparsable amount. This time
the records live in a FILE. Write `summarize_file(path)` returning a
dict with `"total"` (sum of valid amounts), `"valid"`, `"invalid"`.
Driver:

```python
with open("transactions.txt", "w", encoding="utf-8") as f:
    f.write("coffee,-3.50\nsalary,+2500.00\nrent:bad\n\n-12\nsnacks,+4.20\n")

s = summarize_file("transactions.txt")
print(f"total: {s['total']:.2f}")
print(f"valid: {s['valid']}")
print(f"invalid: {s['invalid']}")
```

**Output:**
```
total: 2500.70
valid: 3
invalid: 3
```

**Constraint:** read by iteration; per-line validation logic is yours
to design (you solved this exact grammar before). `split(",")` +
`try/except ValueError` around `float()` — no `isdigit`, no regex.
`summarize_file` never prints. Blank lines are invalid records.

**Hints (ask one at a time):**
- Strip the line first — the trailing newline would ride along otherwise.
- Your Lesson 8 unpacking trick (`name, amount = ...`) still works on a stripped, split line.
- Counters initialize once, before the loop; the loop now yields lines instead of list items.

In [3]:
def summarize_file(path):
    s = {
        "total": 0,
        "valid": 0,
        "invalid": 0
    }
    
    try:
        with open(path, encoding="utf-8") as f:
            for line in f:
                try:
                    name, amount = (part.strip() for part in line.split(","))
                    if not name:
                        raise ValueError("invalid name")
                    amount = float(amount)
                except ValueError:
                    s["invalid"] += 1
                else:
                    s["valid"] += 1
                    s["total"] += amount
                    
    except FileNotFoundError:
        return f"file {path} not found"
        
    return s

with open("transactions.txt", "w", encoding="utf-8") as f:
    f.write("coffee,-3.50\nsalary,+2500.00\nrent:bad\n\n-12\nsnacks,+4.20\n")

s = summarize_file("transactions.txt")
print(f"total: {s['total']:.2f}")
print(f"valid: {s['valid']}")
print(f"invalid: {s['invalid']}")

# but i have a question. 
# the try block have a lot of code
# which is contradictive with my understanding that the try block should be as least as possible
# but i cant find another way to solve this exercise

total: 2500.70
valid: 3
invalid: 3


## M2 — Filter and copy pipeline

**Type:** [REAL-WORLD]

**Task:** write `copy_errors(src, dst)` copying every line whose FIRST
field equals `ERROR` from src to dst, returning how many lines were
copied. Driver:

```python
with open("sensor.log", "w", encoding="utf-8") as f:
    f.write("INFO boot ok\nERROR disk full\nINFO user login\nERROR timeout\nDEBUG noise\n")

n = copy_errors("sensor.log", "errors.log")
print(f"copied: {n}")
print(load_text("errors.log"), end="")
```

**Output:**
```
copied: 2
ERROR disk full
ERROR timeout
```

**Constraint:** ONE `with` statement manages BOTH files (comma form).
Copied lines keep their original text — write the line you read, not
a rebuilt version. Count returned, never printed.

**Hints (ask one at a time):**
- `open(a) as x, open(b) as y:` — one colon, two handles, both auto-closed.
- Splitting once gives you the first field cheaply — or check how the line begins.
- Writing the original line preserves its newline for free; rebuilding it means re-adding one.

In [58]:
def copy_errors(src, dst):
    copied = 0
    try:
        with open(src, encoding="utf-8") as x, open(dst, "w", encoding="utf-8") as y:
            for line in x:
                if line.split(" ", 1)[0] == "ERROR":
                    y.write(line)
                    copied += 1
    except FileNotFoundError as err:
        return f"file not found: {err}"
    return copied

with open("sensor.log", "w", encoding="utf-8") as f:
    f.write("INFO boot ok\nERROR disk full\nINFO user login\nERROR timeout\nDEBUG noise\n")
    
n = copy_errors("sensor.log", "errors.log")
print(f"copied: {n}")
print(load_text("errors.log"), end="")

copied: 2
ERROR disk full
ERROR timeout


## M3 — Typed config loader

**Type:** [REAL-WORLD]

**Task:** config files hold `key=value` text; your program wants typed
values. Given module-level spec `SPEC = {"retries": int, "rate": float,
"name": str}`, write `load_settings(path)` that parses each non-blank
line (split on the FIRST `=`), strips whitespace off the value, and —
for keys present in SPEC — stores the converted value under that key.
Unknown keys are ignored. Conversion failure raises
`ValueError(f"bad setting: {key}")`. Driver:

```python
with open("app.cfg", "w", encoding="utf-8") as f:
    f.write("retries=3\nrate=0.5\nname=expense-cli\n")

try:
    print(load_settings("app.cfg"))
except ValueError as err:
    print(err)

with open("bad.cfg", "w", encoding="utf-8") as f:
    f.write("retries=three\nrate=0.5\n")

try:
    print(load_settings("bad.cfg"))
except ValueError as err:
    print(err)
```

**Output:**
```
{'retries': 3, 'rate': 0.5, 'name': 'expense-cli'}
bad setting: retries
```

**Constraint:** result dict starts EMPTY (blueprints stay read-only).
Values are stripped BEFORE conversion — `int` and `float` mask stray
newlines, `str` does not. Result keys follow SPEC's order. Function
never prints.

**Hints (ask one at a time):**
- Loop over the file lines, look each key up in SPEC — SPEC decides membership, not the file.
- Which conversion hides a leftover newline, and which one exposes it? That asymmetry is why strip comes first.
- You rebuilt this exact raise-with-key contract in Lesson 8 — the caller already knows how to answer it.

In [59]:
SPEC = {
    "retries": int, 
    "rate": float,
    "name": str
}

def load_settings(path):
    result = {}
    
    with open(path, encoding="utf-8") as f:
        for line in f:
            key, value = line.strip().split("=")
            if key in SPEC:
                try:
                    result[key] = SPEC[key](value)
                except ValueError:
                    raise ValueError(f"bad setting: {key}")
    
    return result

with open("app.cfg", "w", encoding="utf-8") as f:
    f.write("retries=3\nrate=0.5\nname=expense-cli\n")

try:
    print(load_settings("app.cfg"))
except ValueError as err:
    print(err)

with open("bad.cfg", "w", encoding="utf-8") as f:
    f.write("retries=three\nrate=0.5\n")

try:
    print(load_settings("bad.cfg"))
except ValueError as err:
    print(err)

{'retries': 3, 'rate': 0.5, 'name': 'expense-cli'}
bad setting: retries


## H1 — Debug: five bugs in the log analyzer

**Type:** [REAL-WORLD] · debug drill

`analyze` reads a log, counts entries, and collects ERROR messages into
a report file. It was written carelessly. Run it, compare expected vs
observed, find and fix **five** bugs.

**Expected output (after fixes):**
```
processed 5 entries
=== errors ===
disk full
timeout
```

**Observed when run as-is:** a crash. After each fix, re-run — the
next bug reveals itself. Read every traceback's first line.

**Rules:**
- Fix each bug with the MINIMAL change. No rewriting working parts.
- Do not modify the driver block.
- After fixing, explain each bug: expected vs observed behavior, root cause.

**The five crime scenes (families):** resource management (no `with`,
wrong mode) · hidden cursor state · state never initialized ·
unauthorized transformation · whitespace discipline.

**Hints (ask one at a time):**
- Run it. Which path does the first traceback complain about — and was that file ever created?
- Once writing works, the loop reports zero passes. What did the line just before the loop do to the cursor?
- With the loop alive, a name is suddenly undefined. Where do names get created?
- Compare the report's letters and blank lines against expected — what transformations happened between reading and writing?

In [ ]:
def analyze(source_path, error_path):
    source = open(source_path)
    consumed = source.read()
    processed = 0
    for line in source:
        processed += 1
        fields = line.split(" ", 1)
        if fields[0] == "ERROR":
            errors.append(fields[1].upper())
    out = open(error_path)
    out.write("=== errors ===\n")
    for msg in errors:
        out.write(msg + "\n")
    out.close()
    return processed


# --- driver (do not change) ---
with open("sensor2.log", "w", encoding="utf-8") as f:
    f.write("INFO boot\nERROR disk full\nINFO scan\nERROR timeout\nINFO done\n")

count = analyze("sensor2.log", "errors2.txt")
print(f"processed {count} entries")
with open("errors2.txt", "r", encoding="utf-8") as f:
    print(f.read(), end="")

In [50]:
def analyze(source_path, error_path):
    errors = []
    processed = 0
    with open(source_path, encoding="utf-8") as source:
        for line in source:
            processed += 1
            fields = line.strip().split(" ", 1)
            if fields[0] == "ERROR":
                errors.append(fields[1])
                
    with open(error_path, "w", encoding="utf-8") as out:
        out.write("=== errors ===\n")
        for msg in errors:
            out.write(msg + "\n")
            
    return processed


# --- driver (do not change) ---
with open("sensor2.log", "w", encoding="utf-8") as f:
    f.write("INFO boot\nERROR disk full\nINFO scan\nERROR timeout\nINFO done\n")

count = analyze("sensor2.log", "errors2.txt")
print(f"processed {count} entries")
with open("errors2.txt", "r", encoding="utf-8") as f:
    print(f.read(), end="")
    
# FileNotFoundError errors2.txt
# add with block to source path open
# delete out.close
# add with block to error path open
# delete consumed source read
# add empty list
# delete upper()
# strip for the fields var

processed 5 entries
=== errors ===
disk full
timeout


## H2 — Interview drill: tail

**Type:** [INTERVIEW]

**Task:** implement `tail(path, n)` returning the last `n` lines of a
file as a list, newlines stripped. If the file has fewer than `n`
lines, return them all; an empty file yields `[]`. Driver:

```python
def make_fixture(path, lines):
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + ("\n" if lines else ""))

make_fixture("app.log", ["boot", "scan", "alert", "fix", "done"])
print(tail("app.log", 2))
print(tail("app.log", 10))
make_fixture("empty.log", [])
print(tail("empty.log", 3))
```

**Output:**
```
['fix', 'done']
['boot', 'scan', 'alert', 'fix', 'done']
[]
```

**Constraint:** TWO PASSES over the file are allowed and expected:
first count lines, then collect the last n while iterating. No
`read()` of the whole file. Newlines stripped from kept lines only.
Function never prints.

**Hints (ask one at a time):**
- Pass one answers "how many total?" — pass two answers "which indices survive?"
- During pass two, enumerate pairs each line with its position — compare position against the cutoff.
- The empty file must fall out naturally from your counting pass — no special case needed if the math is right.

In [64]:
def tail(path, n):
    with open(path, encoding="utf-8") as f:
        total = sum(1 for _ in f)
    
    cutoff = total - n
    result = []
    
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= cutoff:
                result.append(line.strip())
    
    return result
        
    
def make_fixture(path, lines):
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + ("\n" if lines else ""))

make_fixture("app.log", ["boot", "scan", "alert", "fix", "done"])
print(tail("app.log", 2))
print(tail("app.log", 10))
make_fixture("empty.log", [])
print(tail("empty.log", 3))

['fix', 'done']
['boot', 'scan', 'alert', 'fix', 'done']
[]


## Challenge — Expense Persistence CLI

**Type:** [REAL-WORLD] · micro build

Your Expense Validator grows a memory. It loads existing expenses from
a file, merges newly submitted entries, saves everything back
canonically, and reports. Minimum four functions:

- `parse_entry(line)` — same contract as Lesson 8: `description,amount`;
  violations raise `ValueError` (`missing comma` / `empty description` /
  `bad amount`); returns stripped description + float amount. Retype it
  fresh — muscle memory, no copy-paste.
- `load_expenses(path)` — reads stored entries into a list of
  `(description, amount)` tuples; a MISSING file is a normal first run:
  return `[]`.
- `save_expenses(path, expenses)` — rewrites the file canonically: one
  `description,amount` line per entry, amounts formatted `:.2f`
  (normalizing away `+` signs and padding).
- `merge_and_report(path, new_lines, out=print)` — orchestrates: load,
  parse+merge new entries (rejects counted, not printed), save merged,
  print the report. ALL exception handling lives here.

Driver:

```python
with open("expenses.txt", "w", encoding="utf-8") as f:
    f.write("coffee,-3.50\nsalary,+2500.00\n")

new_entries = ["bus,-2.75", "", "snack,abc", "tea, 1.20"]
merge_and_report("expenses.txt", new_entries)

print("--- file now ---")
print(open("expenses.txt", encoding="utf-8").read(), end="")
```

**Output:**
```
=== loaded: 2 ===
accepted: 2
rejected: 2
=== saved: 4 ===
total: 2494.95
--- file now ---
coffee,-3.50
salary,2500.00
bus,-2.75
tea,1.20
```

**Constraints:**
- All file access via `with` + explicit utf-8.
- `load_expenses`/`save_expenses` never print; `merge_and_report` never touches raw parsing itself.
- Exactly one try/except site, inside `merge_and_report`.
- Report lines exactly as shown; `total` covers ALL saved entries, formatted `.2f`.
- Re-running the driver reproduces identical output (idempotent rewrite, no duplication).

**Acceptance criteria:** runs clean on the given input AND on a deleted
`expenses.txt` (first run: `loaded: 0`, file created fresh); every
function does one job.

**Stretch goals (optional):**
- Reject duplicate descriptions during merge (case-insensitive), counted separately as `duplicates`.
- Sort saved entries by amount ascending.
- Make the storage format survive descriptions containing commas (quote them) — sketch only, no parser required.

**Hints (ask one at a time):**
- Data flow first: what lists exist between load and save, and who owns each?
- Canonicalization happens at SAVE time — loading trusts whatever the file holds.
- The missing-file case is load's decision alone; merge shouldn't know files can be absent.

In [74]:
def parse_entry(line: str):
    if "," not in line:
        raise ValueError("missing comma")
    
    description, amount = (part.strip() for part in line.split(","))
    
    if not description:
        raise ValueError("empty description")
    
    try:
        amount = float(amount)
    except ValueError:
        raise ValueError("bad amount")
    
    return description, amount

def load_expenses(path):
    with open(path, encoding="utf-8") as f:
        return [parse_entry(line) for line in f]

def save_expenses(path, expenses):
    formatted = []
    for expense in expenses:
        description, amount = expense
        formatted.append(f"{description},{amount:.2f}\n")
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(formatted)
            
        
def merge_and_report(path, new_lines, out=print):
    result = {
        "loaded": 0,
        "rejected": 0,
        "accepted": 0,
        "saved": 0,
        "total": 0,
    }
    try:
        existed_lines = load_expenses(path)
    except FileNotFoundError:
        expenses = []
    else:
        expenses = [x for x in existed_lines]
    
    for line in new_lines:
        try: 
            description, amount = parse_entry(line)
        except ValueError:
            result["rejected"] += 1
        else:
            result["accepted"] += 1
            expenses.append((description,amount))
    
    save_expenses(path, expenses)
    result["loaded"] = len(existed_lines)
    result["saved"] = len(expenses)
    result["total"] = sum(expense[1] for expense in expenses)

    out(f"=== loaded: {result["loaded"]} ===")
    out(f"accepted: {result["accepted"]}")
    out(f"rejected: {result["rejected"]}")
    out(f"=== saved: {result["saved"]} ===")
    out(f"total: {result["total"]}")
    return


with open("expenses.txt", "w", encoding="utf-8") as f:
    f.write("coffee,-3.50\nsalary,+2500.00\n")

new_entries = ["bus,-2.75", "", "snack,abc", "tea, 1.20"]
merge_and_report("expenses.txt", new_entries)

print("--- file now ---")
print(open("expenses.txt", encoding="utf-8").read(), end="")

=== loaded: 2 ===
accepted: 2
rejected: 2
=== saved: 4 ===
total: 2494.95
--- file now ---
coffee,-3.50
salary,2500.00
bus,-2.75
tea,1.20
